In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.impute import KNNImputer
from functools import reduce
from pathlib import Path
from dotenv import load_dotenv
import os
import re
import warnings
warnings.filterwarnings('ignore')
pd.options.display.max_columns = None
from IPython.display import display
from datetime import datetime
from itertools import combinations
from pandas.api.types import is_integer_dtype
from collections import Counter, defaultdict
from math import ceil

In [ ]:
# Reset cells output
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.max_colwidth')
pd.reset_option('display.width')
pd.reset_option('display.expand_frame_repr')

In [ ]:
print("hello")

In [ ]:
# Define the path to the processed data based on the current user
current_user = os.getlogin()

# Define output directory (notebook is in ./notebooks; data is ../data)
DATA_DIR     = Path(rf"/home/{current_user}/local-share")
OUT_DIR      = DATA_DIR / "processed"

cleaned_path = OUT_DIR / "cleaned_original.csv"

# Let pandas detect the delimiter
data = pd.read_csv(cleaned_path, sep=None, engine="python", encoding="utf-8-sig")
print(data.shape)
data


In [ ]:
data.columns

# Handle low frequency categories in columns

sdo5_degree_degree

sdo1_previous_Previous_Education_Type

sdo2_skc_ADVIES_DEF

  (see cleaning data script for detalis)

In [ ]:
# -----------------------------------------------------------------------------
# Handling Categorical Variables for Mixed Model Families
#
# Purpose
#   Apply a frequency cutoff to merge rare categories into "Other" for selected
#   categorical columns. This reduces noise, prevents overfitting from tiny
#   groups, and avoids one-hot encoding explosion while keeping features
#   compatible across models (LR, SVM → one-hot later; RF → benefits from lower
#   cardinality).
#
# Cutoff and Calculation
#   Chosen cutoff: 0.5% of the dataset (p = 0.005).
#   For this dataset:
#       N = 47,953 rows
#       minimum_count = N × p = 47,953 × 0.005 ≈ 240
#   Any category with < ~240 rows is replaced by "Other".
#
# Why 0.5%?
#   • Cross-validation stability: in 5-fold CV, ~240 total ⇒ ~48 per fold,
#     sufficient for reliable impurity/gain estimation and to avoid empty/micro
#     leaves.
#   • Bias-variance balance: removes idiosyncratic, near-unique levels that
#     encourage overfitting while retaining meaningful categories.
#   • Computational efficiency: limits effective cardinality, speeding up model
#     fitting and preventing one-hot feature explosion in linear/SVM pipelines.
#   • Pragmatic middle ground: lower cutoffs (e.g., 0.1%) admit many unstable
#     levels; higher (e.g., 1%) may discard useful detail.
#
# Notes
#   • NaN values are preserved (not merged into "Other").
#   • Only the specified columns are processed.
#   • If CV folds change, the cutoff can be recalculated as:
#       p = (min_per_fold * n_folds) / N
# -----------------------------------------------------------------------------


# Columns to apply the cutoff to
cat_cols = [
    "sdo5_degree_degree",
    "sdo1_previous_Previous_Education_Type",
    "sdo2_skc_ADVIES_DEF",
]

N = 47953
p = 0.005
min_count = ceil(N * p)  # ≈ 240

for col in cat_cols:
    vc = data[col].value_counts(dropna=False)
    # Exclude NaN from replacement logic; only merge labeled rare levels
    rare_levels = [lvl for lvl in vc.index if (not pd.isna(lvl)) and (vc[lvl] < min_count)]
    if rare_levels:
        data[col] = data[col].where(~data[col].isin(rare_levels), "Others")


In [ ]:
print(data['sdo5_degree_degree'].value_counts())

In [ ]:
print(data['sdo1_previous_Previous_Education_Type'].value_counts())

In [ ]:
print(data['sdo2_skc_ADVIES_DEF'].value_counts())

In [ ]:
data.columns

In [ ]:
# Save version 2 data here to later combine it with version 1
out_path = OUT_DIR / "handled_low_frequency_categories.csv"
data.to_csv(out_path, index=False)